# From text to tokens: a preprocessing pipeline you can measure
Week 4 · Web & Text Analytics

All reviews are **real**, from the *Amazon Unlocked Mobile* dataset (Kaggle, PromptCloud): the same data as last week, now at scale.

| Part | Task | Who writes the code |
|---|---|---|
| **1. Worked example** | Clean, tokenize and filter 2% of the reviews, and measure what every step does | We do, step by step |
| **2. Your turn** (homework) | The same on a 10% sample in your own GitHub repository, with your own decisions, saved as the corpus for weeks 5 and 6 | You do |

In every step we judge a decision the same way: **what happens to the vocabulary, and what information do we lose?**

Run the cells from top to bottom (Shift+Enter).

## The problem
The company has **400,000 phone reviews** and wants, by next week, a model that tells whether a review is **positive or negative**.

A model does not read text: it **counts words**. So before any model, the text has to become something countable.
**Today's job is to prepare the text, not to build the model.**

### The story of this notebook, one line per step
| Step | Question | What we do |
|---|---|---|
| 1 | What exactly do we predict? | Turn the stars into labels |
| 2 | How do we keep the final exam honest? | Hide the test set *before* reading anything |
| 3 | What dirt is in the data? | Measure each kind of noise |
| 4 | What can we safely clean? | Light cleaning → `text_light` |
| 5 | What is a "word"? | Compare four tokenizers |
| 6 | How many words, and which? | A heavy head and a long tail |
| 7 | Can we drop the frequent, useless words? | Stop words, **but not the negations** |
| 8 | Can we merge variants of the same word? | Stemming vs lemmatization |
| 9 | Which combination is best? | All 18 pipelines in one table |


In [1]:
#@title ▶ Run me first: libraries and NLTK data
!pip -q install kagglehub tiktoken

import re, html, unicodedata, glob, os, warnings
warnings.filterwarnings("ignore", message="This pattern is interpreted as a regular expression")
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import nltk
for pkg in ["punkt_tab", "stopwords", "wordnet"]:
    nltk.download(pkg, quiet=True)
from nltk.tokenize import word_tokenize, TweetTokenizer
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import FreqDist
import tiktoken

SEED = 42
print("Ready.")

Ready.


In [4]:
#@title ▶ Load the reviews (about 400,000 rows, 1–2 minutes)
# Option A: download from Kaggle. Option B: upload Amazon_Unlocked_Mobile.csv to Colab (Files panel) and it is used instead.
if os.path.exists("Amazon_Unlocked_Mobile.csv"):
    csv_path = "Amazon_Unlocked_Mobile.csv"
else:
    import kagglehub
    folder = kagglehub.dataset_download("PromptCloudHQ/amazon-reviews-unlocked-mobile-phones")
    csv_path = glob.glob(os.path.join(folder, "*.csv"))[0]

df = pd.read_csv(csv_path)
df.index.name = "review_id"          # the original row number identifies a review in every later week
print(df.shape)
df.head(3)

100%|██████████| 32.6M/32.6M [00:00<00:00, 99.2MB/s]

Extracting files...


(413840, 6)


,Product Name,Brand Name,Price,Rating,Reviews,Review Votes
review_id,,,,,,
0,"""CLEAR CLEAN ESN"" Sprint EPIC 4G Galaxy SPH-D7...",Samsung,199.99,5,I feel so LUCKY to have found this used (phone...,1.0
1,"""CLEAR CLEAN ESN"" Sprint EPIC 4G Galaxy SPH-D7...",Samsung,199.99,4,"nice phone, nice up grade from my pantach revu...",0.0
2,"""CLEAR CLEAN ESN"" Sprint EPIC 4G Galaxy SPH-D7...",Samsung,199.99,5,Very pleased,0.0


In [3]:
#@title ▶ Helper functions
NEGATIONS = {"not", "no", "nor", "never", "n't", "don't", "won't", "didn't", "doesn't", "isn't", "wasn't", "can't", "cannot"}
SW = set(stopwords.words("english"))
SW_REVIEWED = SW - {"not", "no", "nor", "against", "but", "very", "too", "only", "off", "few", "more", "most"}
porter, wnl = PorterStemmer(), WordNetLemmatizer()
WORDS = set(wordnet.all_lemma_names())

def word_tokens(text):
    # word_tokenize, keeping only tokens that contain a letter or digit
    return [t for t in word_tokenize(text) if re.search(r"\w", t)]

def vocab_size(token_lists):
    return len(set(t for toks in token_lists for t in toks))

def run_pipeline(token_lists, lower=False, stop="none", reduce="none"):
    # Apply one configuration to already tokenized reviews; return the new token lists
    out = []
    for toks in token_lists:
        if lower:
            toks = [t.lower() for t in toks]
        if stop == "nltk":
            toks = [t for t in toks if t.lower() not in SW]
        elif stop == "reviewed":
            toks = [t for t in toks if t.lower() not in SW_REVIEWED]
        if reduce == "porter":
            toks = [porter.stem(t, to_lowercase=False) for t in toks]
        elif reduce == "lemma":
            toks = [wnl.lemmatize(t) for t in toks]
        out.append(toks)
    return out

def negations_lost(before, after):
    # How many reviews had a negation before the step and have none after it
    had = [any(t.lower() in NEGATIONS for t in b) for b in before]
    has = [any(t.lower() in NEGATIONS for t in a) for a in after]
    return sum(h and not k for h, k in zip(had, has))

print("Helpers ready.")

Helpers ready.


---
# Part 1 · Worked example: 2% of the reviews
We build the pipeline from the slides on a small sample, so every cell runs in seconds, and we **measure** each step.

## Step 1 · Sample and label
The star rating gives us a label for free. 3-star reviews are ambiguous, so we drop them; 4–5 stars become 1, 1–2 stars become 0.

**Why this step?** A model learns from examples with answers (labels), and nobody will label 400,000 reviews by hand.
The stars already tell us how the customer felt, so they are free labels. 3 stars could go either way, so we leave them out rather than guess.

We work on **2%** of the data so that every cell runs in seconds. In Part 2 you scale the same recipe to 10%.

In [5]:
small = df.sample(frac=0.02, random_state=SEED)
small = small.dropna(subset=["Reviews"])
small = small[small["Rating"] != 3].copy()
small["label"] = (small["Rating"] >= 4).astype(int)

print(len(small), "reviews")
small["label"].value_counts(normalize=True).round(3)

7633 reviews


,proportion
label,
1,0.746
0,0.254


**What the numbers say:** 7,633 reviews, **74.6% positive**. Remember this number: a model that always says “positive” already gets about 75% accuracy.
This imbalance will come back in week 5, when we choose how to evaluate the classifier, and it is the reason the next step uses `stratify=`.

## Step 2 · Split before you read
We split **before** we look at the text. Every decision from now on (which noise to remove, which stop words to keep) is made by reading the training set only.

**Why this step?** At the end we want to know how the model does on reviews it has **never seen**.
For that exam to be honest, *we* must not have seen them either: if we read the test reviews, our cleaning decisions would be shaped by them, and the final score would be optimistic.
So 25% goes into the drawer now, and from here on **every decision comes from `train` only**.

In [6]:
train, test = train_test_split(small, test_size=0.25, random_state=SEED, stratify=small["label"])
print(len(train), "train ·", len(test), "test")
print(train["label"].mean().round(3), test["label"].mean().round(3), "← same share of positives: stratify=")

5724 train · 1909 test
0.746 0.745 ← same share of positives: stratify=


**What the numbers say:** 5,724 train · 1,909 test, with 74.6% and 74.5% positives. `stratify=` kept the same balance in both parts, so the drawer is a fair miniature of the whole.

## Step 3 · Read the data: what noise is there?
Last week's tool: a regex per kind of noise. What share of the training reviews contains each one?

**Why this step?** We do not assume what the noise is; we **measure** it. Only then do we know what is worth cleaning in *this* data.

In [7]:
NOISE = {
    "link":            r"https?://\S*[\w/]|www\.\S+",
    "HTML entity":     r"&#?\w+;",
    "repeated letter": r"(\w)\1{3,}",
    "ALL CAPS word":   r"\b[A-Z]{4,}\b",
    "repeated !?.":    r"([!?.])\1{2,}",
}
rows = []
for name, pat in NOISE.items():
    hit = train["Reviews"].str.contains(pat, regex=True)
    example = train.loc[hit, "Reviews"].iloc[0][:90] if hit.any() else ""
    rows.append((name, round(100 * hit.mean(), 2), example))
pd.DataFrame(rows, columns=["noise", "% of reviews", "first example"])

,noise,% of reviews,first example
0,link,0.19,http://www.amazon.com/gp/product/B013YDFH3Y?re...
1,HTML entity,0.00,
2,repeated letter,0.51,si levanta 3g en venezuela os telefonos me lle...
3,ALL CAPS word,8.21,This phone is pretty great. I brought it becau...
4,repeated !?.,10.15,"From iphone 5 to Galaxy S5, then returned S5, ..."


**What the numbers say:**
* Repeated punctuation (**10.2%**) and ALL-CAPS words (**8.2%**) are the common ones. Links (0.2%) and elongated words like “sooooo” (0.5%) are rare.
* HTML entities: **0%** in this sample. The slides list them as typical noise; our data says they are not a problem here. That is the point of measuring.
* Notice that “!!!” and “USELESS” carry **emotion**. Are they noise or signal? We do not remove them in the next step, and that choice matters later.

## Step 4 · Clean and normalise: `text_light`
Only what helps **every** model: decode HTML entities, remove links, normalise Unicode to NFC, collapse whitespace. Nothing that changes the words themselves.

**Why this step?** We do not know yet which model will use the text, so we clean only what is useless for **every** model: HTML codes, links, odd encodings, extra spaces.
The words stay untouched. The result, `text_light`, is clean but intact.

In [8]:
URL = re.compile(r"https?://\S*[\w/]|www\.\S+")

def clean_light(text):
    text = html.unescape(text)                      # &#34; → "   &amp; → &
    text = URL.sub(" ", text)                       # links carry no meaning
    text = unicodedata.normalize("NFC", text)       # one encoding per character
    return re.sub(r"\s+", " ", text).strip()

train["text_light"] = train["Reviews"].map(clean_light)
changed = train[train["text_light"] != train["Reviews"]]
print(f"{len(changed)} of {len(train)} reviews changed")
for before, after in changed[["Reviews", "text_light"]].head(10).values:
    print("BEFORE:", before[:120]); print("AFTER: ", after[:120]); print()

21 of 5724 reviews changed
BEFORE: http://www.amazon.com/gp/product/B013YDFH3Y?redirect=true&ref_=cm_cr_ryp_prd_ttl_sol_0
AFTER:  

BEFORE: Phone worked fine for a short while then screen went black/blank and unresponsive for no apparent reason. Phone was not 
AFTER:  Phone worked fine for a short while then screen went black/blank and unresponsive for no apparent reason. Phone was not 

BEFORE: This new phone completely worth its money. Within two months of using any defects were not found. Excellent interacts wi
AFTER:  This new phone completely worth its money. Within two months of using any defects were not found. Excellent interacts wi

BEFORE: It's a good phone. I have it set up with a straight talk sim. It's pretty fast and gets the job done. I have a few games
AFTER:  It's a good phone. I have it set up with a straight talk sim. It's pretty fast and gets the job done. I have a few games

BEFORE: Good morning.  Quick Delivery. Excellent delivery. Good cellular Android. I recomme

**What the numbers say:** only **21 of 5,724** reviews changed, almost all because of extra whitespace. Light cleaning is deliberately conservative.
One review was nothing but a link and is now **empty**: a reminder that every cleaning rule can create new edge cases.

## Step 5 · Tokenize: four tokenizers on the same reviews
Vocabulary size and tokens per review for each tokenizer. The GPT-4o tokenizer (tiktoken, `o200k_base`) is the one you pay for when you send text to an LLM.

**Why this step?** The model counts “words”, but what is a word? Is `phone.` the same as `phone`? Is `don't` one word or two?
The tokenizer answers these questions for us, so we compare four of them on the same reviews.

In [9]:
enc = tiktoken.get_encoding("o200k_base")
tweet_tok = TweetTokenizer()
TOKENIZERS = {
    "split()":         lambda t: t.split(),
    "word_tokenize":   word_tokenize,
    "TweetTokenizer":  tweet_tok.tokenize,
    "GPT-4o (o200k)":  lambda t: [enc.decode([i]) for i in enc.encode(t)],
}
rows, tokenized = [], {}
for name, tok in TOKENIZERS.items():
    lists = [tok(t) for t in train["text_light"]]
    tokenized[name] = lists
    rows.append((name, vocab_size(lists), round(sum(map(len, lists)) / len(lists), 1)))
pd.DataFrame(rows, columns=["tokenizer", "vocabulary", "tokens per review"])

,tokenizer,vocabulary,tokens per review
0,split(),21432,38.5
1,word_tokenize,14309,43.8
2,TweetTokenizer,13752,43.5
3,GPT-4o (o200k),13661,45.7


**What the numbers say:** `split()` produces **21,432** different tokens, `word_tokenize` **14,309**. The same text, a third fewer “words”.

**First big lesson:** before any cleaning, the tokenizer has already decided how many words we have. The check below shows where the extra 7,000 come from.

In [12]:
# What would one LLM pass over the training set cost? Look up today's price per 1M input tokens for the model you would use.
PRICE_PER_1M = 2.5
n_tokens = sum(len(enc.encode(t)) for t in train["text_light"])
print(f"{n_tokens:,} GPT-4o tokens in {len(train):,} training reviews")
if PRICE_PER_1M:
    print(f"≈ ${n_tokens / 1e6 * PRICE_PER_1M:,.2f} for one pass, input only")

261,616 GPT-4o tokens in 5,724 training reviews
≈ $0.65 for one pass, input only


**What the numbers say:** an LLM tokenizer is also the **bill**. You pay per token: about 46 tokens per review, roughly \$0.65 for one pass over 5,724 reviews.
Scaled to the full 400,000 reviews, that is in the tens of dollars per pass. Check the current price before you quote it.

From here on we continue with `word_tokenize`, the tokenizer of a count-based model.

---
## Steps 6–9 answer one question
> **We have too many different words. How do we reduce them without losing meaning?**

Step 6 is the diagnosis, steps 7 and 8 are two treatments, step 9 compares every combination of treatments.

## Step 6 · Frequencies: who is on top?
From here on we work with `word_tokenize`, the tokenizer of a count-based model.

**Why this step?** Before reducing anything, we look at what we have: which words, how often.

In [13]:
toks = [word_tokens(t) for t in train["text_light"]]
fd = FreqDist(t.lower() for doc in toks for t in doc if t.isalpha())
hapax = sum(1 for c in fd.values() if c == 1)
print(f"{fd.N():,} word tokens · {len(fd):,} different words · {hapax:,} ({100 * hapax / len(fd):.0f}%) occur once")
print(fd.most_common(15))

# How many of the most frequent words make up half of all tokens?
from itertools import accumulate
cum = list(accumulate(c for _, c in fd.most_common()))   # running total, from the top word down
half = sum(1 for t in cum if t < fd.N() / 2) + 1
print(f"{half} words cover half of all {fd.N():,} word tokens")

213,863 word tokens · 8,849 different words · 4,137 (47%) occur once
[('the', 10296), ('i', 8017), ('it', 6908), ('and', 6068), ('phone', 5931), ('a', 5158), ('to', 5147), ('is', 4333), ('this', 3358), ('for', 2854), ('with', 2451), ('of', 2314), ('my', 2307), ('not', 2232), ('was', 2231)]
53 words cover half of all 213,863 word tokens


**What the numbers say:** the counts show two extremes, and each needs a different treatment.

* **The head.** A handful of words (`the`, `i`, `it`, `and`…) appear everywhere. The 15 most frequent words alone are about **a third of all tokens**, and most say nothing about sentiment. The last line of the output takes this further: rank the words by count, add their counts from the top, and see after how many words the running total reaches **half** of all tokens (on the product reviews of the slides it was 65).
* **The tail.** **47%** of the different words occur only once: typos, rare words, and variants of the same word (`battery` / `batteries`).
  


## Step 7 · Stop words: what do we lose?

**Why this step?** There is a ready-made list of words that “carry no meaning” (stop words). We apply it, and we measure what it removes.

In [ ]:
lower_toks = run_pipeline(toks, lower=True)
for name, stop in [("NLTK list", "nltk"), ("reviewed list", "reviewed")]:
    after = run_pipeline(lower_toks, stop=stop)
    print(f"{name:14s} vocabulary {vocab_size(after):6,}   negations lost in {negations_lost(lower_toks, after):,} reviews")

# Three reviews whose negation disappears with the unedited list
after = run_pipeline(lower_toks, stop="nltk")
shown = 0
for b, a, text in zip(lower_toks, after, train["text_light"]):
    if any(t in NEGATIONS for t in b) and not any(t in NEGATIONS for t in a) and len(b) < 15:
        print("\n", text, "\n →", a); shown += 1
    if shown == 3: break

NLTK list      vocabulary 11,592   negations lost in 1,192 reviews
reviewed list  vocabulary 11,604   negations lost in 0 reviews

 The USB connection did not work 
 → ['usb', 'connection', 'work']

 WiFi did not connect right out the box. 
 → ['wifi', 'connect', 'right', 'box']

 The phone case would not close after phone was inserted.Just did not fit. 
 → ['phone', 'case', 'would', 'close', 'phone', 'inserted.just', 'fit']


**What the numbers say:**
* NLTK's list contains `not`, `no`, `nor`. **1,192 training reviews (about 1 in 5)** lose every negation they had.
* “did not work” becomes `work`. **The review now says the opposite.**
* The vocabulary barely moves (11,727 → 11,592, about 135 types): as step 6 predicted, stop words are a head treatment, not a vocabulary treatment.

**This is the heart of the notebook:** the standard recipe can destroy exactly the information our task needs. The fix is to edit the list. The reviewed list gives the negations back, loses **0** reviews, and costs only 12 extra vocabulary entries.

### ✋ Your move · Your own stop-word list (4 minutes)
Give words back from NLTK's list so that **no review loses its negation**, while removing as few words from the list as possible.


In [ ]:
SW_MINE = SW - set()     # TODO: put inside set() the words you give back, e.g. {"word1", "word2"}

In [ ]:
#@title ▶ Check (run it, do not edit)
mine = [[t for t in doc if t not in SW_MINE] for doc in lower_toks]
lost = negations_lost(lower_toks, mine)
given_back = sorted(SW - SW_MINE)
print(f"words given back: {len(given_back)} {given_back}")
print(f"vocabulary {vocab_size(mine):,} · reviews that lose their negation: {lost}")
if lost == 0:
    print(f"✓ no negation lost, giving back {len(given_back)} word(s). Could fewer do it?")
else:
    for before, after_ in zip(lower_toks, mine):
        if any(t in NEGATIONS for t in before) and not any(t in NEGATIONS for t in after_):
            print("✗ for example:", " ".join(before), "\n             →", " ".join(after_))
            break

*Hint if you are stuck:* look at how `word_tokenize` splits “don't”, and which of the resulting tokens are in `NEGATIONS`.

## Step 8 · Stemming or lemmatization?

**Why this step?** `battery` and `batteries` mean the same thing, but the model sees two different words. There are two ways to merge them:
* **Stemming** (Porter) cuts endings with a knife: `batteries` → `batteri`. It reduces a lot, but produces non-words and sometimes joins unrelated words.
* **Lemmatization** (WordNet) looks the word up in a dictionary: `batteries` → `battery`. It reduces less, but safely.

Once again a trade-off: how much reduction do we want, and how much precision do we give up for it?

In [ ]:
for reduce in ["none", "porter", "lemma"]:
    out = run_pipeline(lower_toks, stop="reviewed", reduce=reduce)
    nonwords = {t for doc in out for t in doc if t.isalpha() and t not in WORDS and t not in SW}
    print(f"{reduce:7s} vocabulary {vocab_size(out):6,}   types not in WordNet: {len(nonwords):,}")

stems = Counter(porter.stem(t) for doc in lower_toks for t in doc if t.isalpha())
print("\nMost frequent Porter stems that are not words:",
      [s for s, _ in stems.most_common(300) if s not in WORDS and s not in SW][:12])

none    vocabulary 11,604   types not in WordNet: 3,807
porter  vocabulary  9,206   types not in WordNet: 3,585
lemma   vocabulary 10,867   types not in WordNet: 3,001

Most frequent Porter stems that are not words: ['thi', 'veri', 'would', 'batteri', 'onli', 'realli', 'bought', 'sim', 'app', 'got', 'purchas', 'becaus']


**What the numbers say:**
* Porter: 11,604 → **9,206** (−21%), but the most frequent stems are non-words: `thi`, `veri`, `batteri`, `onli`, `realli`.
* Lemmatization: 11,604 → **10,867** (−6%).


## Step 9 · One table, every pipeline
The same comparison as step 9 of the interactive exercise, now on thousands of reviews.

**Why this step?** We have three decisions (lowercase or not, which stop-word list, stemming / lemma / nothing), so **2 × 3 × 3 = 18** pipelines.
We run them all and keep two numbers for each: **how many words remain** and **how many negations were lost**. This table is the “decision with numbers”.

In [ ]:
rows = []
for lower in [False, True]:
    for stop in ["none", "nltk", "reviewed"]:
        for reduce in ["none", "porter", "lemma"]:
            out = run_pipeline(toks, lower, stop, reduce)
            rows.append((lower, stop, reduce, vocab_size(out), negations_lost(toks, out)))
table = pd.DataFrame(rows, columns=["lowercase", "stop words", "reduce", "vocabulary", "negations lost"])
table.sort_values("vocabulary")

,lowercase,stop words,reduce,vocabulary,negations lost
13,True,nltk,porter,9194,1192
16,True,reviewed,porter,9206,0
10,True,none,porter,9299,0
14,True,nltk,lemma,10855,1192
17,True,reviewed,lemma,10867,0
11,True,none,lemma,10975,0
4,False,nltk,porter,11385,1192
7,False,reviewed,porter,11416,0
12,True,nltk,none,11592,1192
15,True,reviewed,none,11604,0
